In [1]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [medmnist]


In [2]:
"""
DCGAN on DermaMNIST (skin-lesion images) -- a small learning project.

Dataset : DermaMNIST from MedMNIST v2/v3 (derived from HAM10000, public, auto-downloaded).
Model   : DCGAN (Radford et al., 2015).
Output  : sample grids per epoch, loss curve, and generator checkpoints in ./outputs
"""
import argparse
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.utils as vutils
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
import medmnist
from medmnist import INFO


class Generator(nn.Module):
    """z (nz x 1 x 1) -> RGB image 64 x 64."""
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),                      # 4x4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),                      # 8x8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),                      # 16x16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),                          # 32x32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),                                                   # 64x64
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """RGB image 64 x 64 -> real/fake logit."""
    def __init__(self, ndf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),                                     # 32x32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),            # 16x16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),            # 8x8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, True),            # 4x4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),                  # 1x1 logit
        )

    def forward(self, x):
        return self.net(x).view(-1)


def weights_init(m):
    name = m.__class__.__name__
    if "Conv" in name:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


def get_loader(args):
    info = INFO["dermamnist"]
    DataClass = getattr(medmnist, info["python_class"])
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3),   # to [-1, 1] for Tanh
    ])
    os.makedirs(args.data_dir, exist_ok=True)
    ds = DataClass(split="train", transform=tfm, download=True, size=64, root=args.data_dir)
    if args.class_id is not None:                       # optionally train on a single lesion class
        idx = [i for i, (_, y) in enumerate(zip(range(len(ds)), ds.labels)) if int(y[0]) == args.class_id]
        ds = Subset(ds, idx)
        print(f"Using class {args.class_id} only: {len(ds)} images")
    else:
        print(f"Using all classes: {len(ds)} images")
    return DataLoader(ds, batch_size=args.batch_size, shuffle=True,
                      num_workers=args.workers, drop_last=True)


def main(args):
    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.out_dir, exist_ok=True)
    print("Device:", device)

    loader = get_loader(args)
    G = Generator(args.nz).to(device); G.apply(weights_init)
    D = Discriminator().to(device);   D.apply(weights_init)

    criterion = nn.BCEWithLogitsLoss()
    optG = torch.optim.Adam(G.parameters(), lr=args.lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=args.lr, betas=(0.5, 0.999))
    fixed_noise = torch.randn(64, args.nz, 1, 1, device=device)

    g_losses, d_losses = [], []
    for epoch in range(1, args.epochs + 1):
        g_run, d_run = 0.0, 0.0
        for real, _ in loader:
            real = real.to(device)
            b = real.size(0)
            real_lbl = torch.full((b,), 0.9, device=device)   # one-sided label smoothing
            fake_lbl = torch.zeros(b, device=device)

            # --- Discriminator step ---
            optD.zero_grad()
            loss_real = criterion(D(real), real_lbl)
            noise = torch.randn(b, args.nz, 1, 1, device=device)
            fake = G(noise)
            loss_fake = criterion(D(fake.detach()), fake_lbl)
            lossD = loss_real + loss_fake
            lossD.backward(); optD.step()

            # --- Generator step ---
            optG.zero_grad()
            lossG = criterion(D(fake), torch.ones(b, device=device))
            lossG.backward(); optG.step()

            g_run += lossG.item(); d_run += lossD.item()

        g_losses.append(g_run / len(loader)); d_losses.append(d_run / len(loader))
        print(f"Epoch {epoch:03d}/{args.epochs} | D loss {d_losses[-1]:.3f} | G loss {g_losses[-1]:.3f}")

        if epoch % args.sample_every == 0 or epoch == args.epochs:
            G.eval()
            with torch.no_grad():
                samples = G(fixed_noise).cpu()
            G.train()
            vutils.save_image(samples, f"{args.out_dir}/samples_epoch_{epoch:03d}.png",
                              nrow=8, normalize=True, value_range=(-1, 1))
            torch.save(G.state_dict(), f"{args.out_dir}/generator.pt")

    plt.figure(figsize=(6, 4))
    plt.plot(g_losses, label="Generator"); plt.plot(d_losses, label="Discriminator")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("DCGAN training loss")
    plt.tight_layout(); plt.savefig(f"{args.out_dir}/loss_curve.png", dpi=150)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--batch_size", type=int, default=128)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--nz", type=int, default=100)
    p.add_argument("--class_id", type=int, default=None, help="train on one DermaMNIST class (0-6)")
    p.add_argument("--sample_every", type=int, default=5)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--data_dir", default="./data")
    p.add_argument("--out_dir", default="./outputs")
    main(p.parse_args(args=[]))


Device: cuda


100%|██████████| 100M/100M [00:29<00:00, 3.37MB/s]


Using all classes: 7007 images
Epoch 001/50 | D loss 0.888 | G loss 10.778
Epoch 002/50 | D loss 0.780 | G loss 5.080
Epoch 003/50 | D loss 0.553 | G loss 7.053
Epoch 004/50 | D loss 0.445 | G loss 5.929
Epoch 005/50 | D loss 0.711 | G loss 5.863
Epoch 006/50 | D loss 0.872 | G loss 3.212
Epoch 007/50 | D loss 0.985 | G loss 2.897
Epoch 008/50 | D loss 0.996 | G loss 2.789
Epoch 009/50 | D loss 1.082 | G loss 2.709
Epoch 010/50 | D loss 1.001 | G loss 2.390
Epoch 011/50 | D loss 1.051 | G loss 2.154
Epoch 012/50 | D loss 1.108 | G loss 2.352
Epoch 013/50 | D loss 1.172 | G loss 2.193
Epoch 014/50 | D loss 1.134 | G loss 2.078
Epoch 015/50 | D loss 1.126 | G loss 1.942
Epoch 016/50 | D loss 1.023 | G loss 2.051
Epoch 017/50 | D loss 1.077 | G loss 2.050
Epoch 018/50 | D loss 1.042 | G loss 2.000
Epoch 019/50 | D loss 1.174 | G loss 2.096
Epoch 020/50 | D loss 1.170 | G loss 1.830
Epoch 021/50 | D loss 1.224 | G loss 1.798
Epoch 022/50 | D loss 1.210 | G loss 1.732
Epoch 023/50 | D loss 

In [3]:
import shutil

# Zip the outputs directory
shutil.make_archive('/content/outputs', 'zip', '/content/outputs')
print("Successfully zipped /content/outputs to /content/outputs.zip")


Successfully zipped /content/outputs to /content/outputs.zip
